# Capstone Projects
Each section is a compact version of one capstone from this folder's README, built from earlier modules. Each one starts from a signal whose ground truth is known (a note's exact pitch, a drum loop's exact beat times, the clean speech before noise was added), so the result can be measured, not just heard.

1. **Instrument tuner**: pitch detection with the normalized autocorrelation (modules 01, 02, 04).
2. **Beat and tempo tracker**: onset detection from spectral flux, tempo from autocorrelation (01, 02, 03).
3. **Speech front end**: voice activity detection, noise reduction and MFCC features (02, 03, 06, 07).
4. **Telephone channel**: band limiting, 8 kHz resampling, mu-law coding, and echo cancellation with double-talk detection (03, 05, 06, 07).
5. **Game audio scene**: a moving sound source rendered for headphones with time and level differences between the ears, distance cues and reverb (03, 04, 05).

Sections 1 to 4 run offline on whole signals, but process them frame by frame, the way a real-time version would. Section 5 updates its parameters block by block, like a game audio engine.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import fft as sp_fft, signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import (add_noise, drum_loop, mu_law_compress, mu_law_expand, nlms, noise_reduction,
                      pluck, synthetic_vowels, time_axis)

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

FS = 44100
rng = np.random.default_rng(0)

def play(x, label, fs=FS):
    print(f'{label}:')
    display.display(display.Audio(np.clip(x, -1, 1), rate=fs, normalize=False))

def snr_db(clean, test):
    return 10 * np.log10(np.mean(clean**2) / np.mean((test - clean)**2))

## 1. Instrument tuner
A tuner estimates the pitch of a note many times per second and shows how far it is from the nearest note of the equal-tempered scale, in **cents** (hundredths of a semitone).

Plain autocorrelation (`01_Discrete_Time_Signals_and_Systems`) often picks the wrong peak: a peak at twice the period is nearly as tall as the true one, and strong harmonics add peaks at fractions of it. The **normalized square difference function** of the McLeod pitch method (McLeod and Wyvill, 2005) fixes most of this:

$$n(\tau) = \frac{2\sum_j x_j x_{j+\tau}}{\sum_j \left(x_j^2 + x_{j+\tau}^2\right)}$$

It lies between -1 and 1, and is close to 1 at every multiple of the period. The detector takes the **first** peak that reaches 90% of the highest one, which is the period rather than a multiple of it, and refines its position by parabolic interpolation, for a resolution far below one sample.

Frames are 2048 samples (46 ms at 44.1 kHz). The lowest guitar string, E2 at 82 Hz, has a 12 ms period, so every frame holds several periods. The test notes are plucked tones detuned by known amounts, with noise added.

In [ ]:
NOTE_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def nearest_note(f):
    midi = 69 + 12 * np.log2(f / 440)
    nearest = int(round(midi))
    return f'{NOTE_NAMES[nearest % 12]}{nearest // 12 - 1}', 100 * (midi - nearest)

def detect_pitch(frame, fs=FS, fmin=60, fmax=1200, key_threshold=0.9, clarity_threshold=0.6):
    """McLeod pitch method. Returns the frequency in Hz, or None if the frame is not clearly pitched."""
    x = frame - frame.mean()
    n = len(x)
    spectrum = np.fft.rfft(x, 2 * n)
    r = np.fft.irfft(np.abs(spectrum)**2)[:n]                       # autocorrelation
    cumulative = np.concatenate(([0.0], np.cumsum(x**2)))
    tau = np.arange(n)
    m = cumulative[n - tau] + (cumulative[n] - cumulative[tau])     # sum of x_j^2 + x_{j+tau}^2
    nsdf = 2 * r / np.maximum(m, 1e-12)
    lo, hi = int(fs / fmax), int(fs / fmin)
    peaks, _ = signal.find_peaks(nsdf[lo:hi])
    if len(peaks) == 0:
        return None
    peaks += lo
    best = nsdf[peaks].max()
    if best < clarity_threshold:
        return None
    p = peaks[np.argmax(nsdf[peaks] >= key_threshold * best)]      # first peak close to the best
    a, b, c = nsdf[p - 1], nsdf[p], nsdf[p + 1]
    period = p + 0.5 * (a - c) / (a - 2 * b + c)
    return fs / period

def track_pitch(x, frame=2048, hop=1024):
    times, pitches = [], []
    for start in range(0, len(x) - frame, hop):
        times.append((start + frame / 2) / FS)
        pitches.append(detect_pitch(x[start:start + frame]))
    return np.array(times), pitches

print(f'{"Played":>8}{"Detuned by":>12}{"Detected":>10}{"Reading":>18}{"Error":>9}')
for note, detune in [(40, 0), (45, 15), (50, -23), (55, 7), (59, -4), (64, 31), (69, 0), (76, -12)]:
    f_true = 440 * 2 ** ((note - 69 + detune / 100) / 12)
    x = add_noise(pluck(f_true, 1.0, FS, decay=0.6), 0.02, rng)
    _, pitches = track_pitch(x[int(0.05 * FS):int(0.6 * FS)])
    f_est = np.median([p for p in pitches if p is not None])
    name, cents = nearest_note(f_est)
    played, _ = nearest_note(440 * 2 ** ((note - 69) / 12))
    print(f'{played:>8}{detune:>8} ct{name:>10}{cents:>+15.1f} ct{cents - detune:>+7.2f} ct')

In [ ]:
def tuner_demo(note=45, detune=15, noise=0.02):
    f_true = 440 * 2 ** ((note - 69 + detune / 100) / 12)
    x = add_noise(pluck(f_true, 2.0, FS, decay=0.6), noise, np.random.default_rng(1))
    times, pitches = track_pitch(x)
    cents = [nearest_note(p)[1] if p is not None else np.nan for p in pitches]
    names = [nearest_note(p)[0] for p in pitches if p is not None]
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    axes[0].plot(np.arange(len(x)) / FS, x, linewidth=0.5)
    axes[0].set_title('Input')
    axes[0].set_xlabel('Time (s)')
    axes[1].plot(times, cents, 'o-')
    axes[1].axhline(detune, color='k', linestyle=':', label=f'true: {detune:+d} cents')
    axes[1].set_ylim(-50, 50)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Cents from nearest note')
    axes[1].set_title(f'Tuner reading (note: {max(set(names), key=names.count) if names else "none"})')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    print('Gaps in the reading are frames the detector judged unpitched: the note has decayed into the noise.')
    play(x, 'Input')

display.display(interactive(
    tuner_demo,
    note=widgets.IntSlider(value=45, min=40, max=84, description='MIDI note:'),
    detune=widgets.IntSlider(value=15, min=-50, max=50, description='Detune (ct):'),
    noise=widgets.FloatSlider(value=0.02, min=0.0, max=0.3, step=0.01, description='Noise:'),
))

## 2. Beat and tempo tracker
Beat tracking has three steps:
1. **Onset strength**: how much new energy appears in each STFT frame. **Spectral flux** sums the increases in log magnitude from one frame to the next, ignoring decreases. It peaks at every drum hit or note start. Only bins below 500 Hz are used, where the kick and snare drums mark the beats. The hi-hats, which play twice per beat, would otherwise suggest double the tempo.
2. **Tempo**: the onset strength repeats with the beat period, so its autocorrelation peaks at that lag, searched between 60 and 180 beats per minute (BPM). Two details matter. The beat period is rarely a whole number of frames, so a sharp peak gets split between two neighboring lags and can lose to a weaker rhythm that happens to fall on a whole number. Smoothing the onset strength first makes the peaks broad enough to survive this. And rhythms repeat at half, double, 1.5 times the beat, and so on, so the autocorrelation is weighted towards tempos near 120 BPM, the most common range (a **tempo prior**). Choosing between related tempos is the classic difficulty of beat tracking.
3. **Beat phase**: slide a grid of beats with that period over the onset strength, and keep the offset where the grid collects the most onset energy.

The test signal is the drum loop from `08_Audio_Synthesis_and_Sound_Design`, played with human-like random timing errors, so the true beat times are known. A detected beat counts as correct if it is within 70 ms of a true one, and the **F-measure** combines the fraction of detected beats that are correct with the fraction of true beats that were found.

In [ ]:
HOP = 512

def onset_strength(x, max_freq=500):
    f, t, X = signal.stft(x, FS, nperseg=2048, noverlap=2048 - HOP)
    log_mag = np.log1p(100 * np.abs(X[f < max_freq]))
    flux = np.sum(np.maximum(np.diff(log_mag, axis=1), 0), axis=0)
    flux -= np.convolve(flux, np.ones(16) / 16, mode='same')           # remove the slowly varying part
    flux = np.maximum(flux, 0)
    smooth = np.exp(-0.5 * (np.arange(-6, 7) / 2.0)**2)                 # Gaussian, 2 frames wide
    return t[1:], np.convolve(flux, smooth / smooth.sum(), mode='same')

def estimate_tempo(onset, frame_rate, bpm_range=(60, 180), prior_bpm=120):
    r = np.correlate(onset, onset, mode='full')[len(onset) - 1:]
    lags = np.arange(int(frame_rate * 60 / bpm_range[1]), int(frame_rate * 60 / bpm_range[0]) + 1)
    prior = np.exp(-0.5 * np.log2(60 * frame_rate / lags / prior_bpm)**2)   # one octave wide, on a log scale
    i = lags[np.argmax(r[lags] * prior)]
    a, b, c = r[i - 1], r[i], r[i + 1]
    period_frames = i + 0.5 * (a - c) / (a - 2 * b + c)
    return 60 * frame_rate / period_frames, period_frames

def beat_times_from(onset, onset_times, period_frames):
    frames = np.arange(len(onset))
    offsets = np.linspace(0, period_frames, 200, endpoint=False)
    scores = [np.interp(np.arange(o, len(onset) - 1, period_frames), frames, onset).sum() for o in offsets]
    best = offsets[np.argmax(scores)]
    return np.interp(np.arange(best, len(onset) - 1, period_frames), frames, onset_times)

def f_measure(detected, truth, tolerance=0.07):
    hits = sum(np.min(np.abs(truth - d)) <= tolerance for d in detected)
    precision, recall = hits / len(detected), hits / len(truth)
    return 2 * precision * recall / (precision + recall) if hits else 0.0

def beat_tracker_demo(bpm=117, jitter_ms=8):
    x, true_beats = drum_loop(bpm, 8, FS, np.random.default_rng(2), timing_jitter=jitter_ms / 1000)
    x = add_noise(x, 0.01, rng)
    onset_times, onset = onset_strength(x)
    frame_rate = FS / HOP
    bpm_est, period_frames = estimate_tempo(onset, frame_rate)
    beats = beat_times_from(onset, onset_times, period_frames)
    print(f'True tempo {bpm} BPM, estimated {bpm_est:.2f} BPM. Beat F-measure: {f_measure(beats, true_beats):.2f}')

    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(onset_times, onset / onset.max(), linewidth=1, label='onset strength')
    ax.vlines(true_beats, 0, 1.1, colors='k', linestyles=':', linewidth=1, label='true beats')
    ax.vlines(beats, 0, 1.1, colors='r', linewidth=1, label='detected beats')
    ax.set_xlim(0, 4)
    ax.set_xlabel('Time (s)')
    ax.legend(loc='upper right')
    plt.show()
    click = np.sin(2 * np.pi * 1500 * time_axis(0.03, FS)) * np.exp(-time_axis(0.03, FS) / 0.005)
    clicks = np.zeros(len(x))
    for b in beats:
        i = int(b * FS)
        clicks[i:i + len(click)] += click[:len(x) - i]
    play(x + 0.5 * clicks, 'Drum loop with clicks on the detected beats')

display.display(interactive(
    beat_tracker_demo,
    bpm=widgets.IntSlider(value=117, min=70, max=170, description='Tempo (BPM):'),
    jitter_ms=widgets.IntSlider(value=8, min=0, max=40, description='Jitter (ms):'),
))

On this drum loop, the tracker finds the tempo to within 0.3 BPM from 70 to 150 BPM. At 160 BPM and above, the prior pulls it to half the tempo: the tracker taps along at every second beat, a mistake people also make with fast music.

## 3. Speech front end
Speech recognizers do not work on raw audio. A front end first reduces each 10 ms of audio to a short feature vector, usually **mel-frequency cepstral coefficients** (MFCCs):
1. **Pre-emphasis**, $y[n] = x[n] - 0.97\,x[n-1]$, which lifts high frequencies, where speech has less energy.
2. Frames of 25 ms, every 10 ms, each with a Hamming window and an FFT.
3. A **mel filterbank**: 26 triangular filters spaced evenly on the mel scale, $\text{mel}(f) = 2595 \log_{10}(1 + f/700)$, which, like the Bark scale in `06_Psychoacoustics_and_Audio_Coding`, follows the frequency resolution of hearing.
4. The logarithm of each filter's energy, then a discrete cosine transform. The first 13 coefficients are the MFCCs.

The rest of the front end protects them from the environment. **Voice activity detection** (VAD) finds the frames that contain speech, here by comparing each frame's energy with the noise floor. **Noise reduction** (`07_Audio_Restoration_and_Enhancement`) cleans the signal first. The test measures how far the MFCCs of noisy and of enhanced speech are from those of the clean speech: the closer, the better a recognizer trained on clean speech would do.

In [ ]:
FS_SPEECH = 16000
clean, active = synthetic_vowels('aeiouieao', sample_rate=FS_SPEECH)
FRAME, HOP_S = 400, 160                                              # 25 ms and 10 ms at 16 kHz

def mel_filterbank(n_filters=26, n_fft=512, fs=FS_SPEECH):
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    edges = hz(np.linspace(mel(0), mel(fs / 2), n_filters + 2))
    bins = np.fft.rfftfreq(n_fft, 1 / fs)
    bank = np.zeros((n_filters, len(bins)))
    for i in range(n_filters):
        lo, center, hi = edges[i:i + 3]
        bank[i] = np.clip(np.minimum((bins - lo) / (center - lo), (hi - bins) / (hi - center)), 0, None)
    return bank

MEL_BANK = mel_filterbank()

def frames_of(x):
    return np.lib.stride_tricks.sliding_window_view(x, FRAME)[::HOP_S]

def mfcc(x, n_coefs=13):
    emphasized = np.append(x[0], x[1:] - 0.97 * x[:-1])
    power = np.abs(np.fft.rfft(frames_of(emphasized) * np.hamming(FRAME), 512))**2
    log_mel = np.log(power @ MEL_BANK.T + 1e-10)
    return sp_fft.dct(log_mel, type=2, norm='ortho', axis=1)[:, :n_coefs]

def vad(x, margin_db=3.0):
    energy_db = 10 * np.log10(np.mean(frames_of(x)**2, axis=1) + 1e-12)
    floor = np.percentile(energy_db, 10)                             # the quietest frames are noise
    return signal.medfilt((energy_db > floor + margin_db).astype(float), 5) > 0

def front_end_demo(snr_in=5):
    noise = rng.normal(0, 1, len(clean))
    noise *= np.sqrt(np.mean(clean[active]**2) / np.mean(noise**2)) * 10 ** (-snr_in / 20)
    noisy = clean + noise
    enhanced = noise_reduction(noisy, noisy[:int(0.4 * FS_SPEECH)], FS_SPEECH, method='wiener')
    truth = frames_of(active).mean(axis=1) > 0.5
    detected = vad(noisy)
    print(f'VAD on noisy speech: {100 * np.mean(detected == truth):.1f}% of frames correct '
          f'({100 * np.mean(detected[truth]):.0f}% of speech frames found, {100 * np.mean(detected[~truth]):.0f}% of silent frames flagged)')
    reference = mfcc(clean)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
    for ax, (name, y) in zip(axes, [('clean', clean), ('noisy', noisy), ('enhanced', enhanced)]):
        features = mfcc(y)
        distance = np.mean(np.linalg.norm((features - reference)[truth, 1:], axis=1))
        ax.imshow(features[:, 1:].T, aspect='auto', origin='lower', cmap='coolwarm', vmin=-15, vmax=15,
                  extent=[0, len(features) * HOP_S / FS_SPEECH, 1, 12])
        ax.set_title(f'MFCC 1-12, {name}' + (f'\ndistance from clean: {distance:.2f}' if name != 'clean' else '\n'))
        ax.set_xlabel('Time (s)')
    axes[0].set_ylabel('Coefficient')
    plt.tight_layout()
    plt.show()
    play(noisy, 'Noisy', FS_SPEECH)
    play(enhanced, 'Enhanced', FS_SPEECH)

display.display(interactive(
    front_end_demo,
    snr_in=widgets.IntSlider(value=5, min=-5, max=25, description='Input SNR (dB):'),
))

An energy-based VAD needs speech to stand out from the noise floor. Below about 0 dB SNR it misses most speech frames. Practical detectors add features that noise lacks, such as periodicity (pitch) or the spectral shape of speech.

## 4. Telephone channel
A phone call passes through several stages, each from an earlier module:
1. **Band limiting** to 300 to 3400 Hz, the traditional telephone band (`03_Digital_Filters`).
2. **Resampling** to 8 kHz (`03_Digital_Filters/resampling.ipynb`).
3. **mu-law coding** at 8 bits, 64 kbit/s (`06_Psychoacoustics_and_Audio_Coding`).
4. **Echo**: at the far end, our own voice leaks back into the line from the far phone's loudspeaker or from the analog line's hybrid circuit, and returns to us delayed. An **echo canceller** at the far end learns the echo path with an NLMS adaptive filter (`07_Audio_Restoration_and_Enhancement`) and subtracts the echo.

The echo canceller has a problem: when the far-end person talks at the same time (**double talk**), their voice is in the error signal, and the filter tries to cancel it too, which wrecks the learned echo path. A **double-talk detector** pauses adaptation when the microphone is louder than the echo could be. The simple **Geigel detector** declares double talk whenever the microphone level exceeds half of the recent peak of our voice, assuming the echo is at least 6 dB weaker than our voice.

Performance is measured as **echo return loss enhancement** (ERLE): how much weaker the echo is after cancellation, measured while only our voice is present.

In [ ]:
FS_TEL = 8000

def telephone_band(x, fs_in=FS_SPEECH):
    band = signal.butter(4, [300, 3400], btype='bandpass', fs=fs_in, output='sos')
    narrow = signal.resample_poly(signal.sosfilt(band, x), FS_TEL, fs_in)
    step = 2 / 256
    coded = mu_law_expand(np.clip(np.round(mu_law_compress(narrow) / step) * step, -1, 1 - step))
    return narrow, coded

our_voice, our_active = synthetic_vowels('aoueiaouei', sample_rate=FS_SPEECH, f0=110, lead_in=0.3)
their_voice, _ = synthetic_vowels('iea', sample_rate=FS_SPEECH, f0=210, lead_in=2.8)
length = max(len(our_voice), len(their_voice))
our_voice = np.pad(our_voice, (0, length - len(our_voice)))
their_voice = np.pad(their_voice, (0, length - len(their_voice)))

_, x_far = telephone_band(our_voice)                 # our voice, as it arrives at the far phone
_, near = telephone_band(their_voice)                # the far-end talker's voice at their microphone
echo_path = np.zeros(256)
echo_path[160:] = rng.normal(0, 1, 96) * np.exp(-np.arange(96) / 20)   # 20 ms delay, then a decaying response
echo_path *= 10 ** (-10 / 20) / np.linalg.norm(echo_path)             # echo 10 dB weaker than our voice
echo = np.convolve(x_far, echo_path)[:len(x_far)]
microphone = near + echo + rng.normal(0, 1e-3, len(x_far))

window = len(echo_path)
recent_peak = np.lib.stride_tricks.sliding_window_view(np.pad(np.abs(x_far), (window - 1, 0)), window).max(axis=1)
no_double_talk = np.abs(microphone) < 0.5 * recent_peak                 # Geigel detector

only_echo = (np.abs(near) < 1e-4) & (recent_peak > 0.01)             # our voice present, far-end talker silent
after_double_talk = only_echo & (np.arange(len(near)) > np.flatnonzero(np.abs(near) > 1e-4).max())

print(f'Echo canceller, 256 taps (32 ms). ERLE while only our voice is present:')
results = {}
for name, adapt in [('no double-talk detector', None), ('Geigel double-talk detector', no_double_talk)]:
    cleaned, _ = nlms(x_far, microphone, n_taps=window, step=0.5, adapt=adapt)
    results[name] = cleaned
    erle = lambda mask: 10 * np.log10(np.mean(echo[mask]**2) / np.mean((cleaned - near)[mask]**2))
    print(f'  {name:<28} overall {erle(only_echo):5.1f} dB, after the double talk {erle(after_double_talk):5.1f} dB')

t_tel = np.arange(len(near)) / FS_TEL
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_tel, microphone, color='0.7', linewidth=0.5, label='far-end microphone: their voice + echo of ours')
ax.plot(t_tel, results['Geigel double-talk detector'] - near, linewidth=0.5, label='echo left after cancellation')
ax.fill_between(t_tel, -1, 1, where=~no_double_talk & (np.abs(near) > 1e-4), color='orange', alpha=0.2, label='adaptation paused (double talk)')
ax.set_ylim(-0.6, 0.6)
ax.set_xlabel('Time (s)')
ax.legend(loc='upper right', fontsize=9)
plt.show()

narrow, coded = telephone_band(our_voice)
play(our_voice, 'Our voice, 16 kHz', FS_SPEECH)
play(coded, 'After band limiting, 8 kHz and mu-law', FS_TEL)
play(microphone, 'What we hear back: their voice plus the echo of ours', FS_TEL)
play(results['Geigel double-talk detector'], 'After echo cancellation', FS_TEL)

Without the detector, the filter adapts during double talk and partly "learns" the other talker's voice instead of the echo path, so the echo comes back afterwards until it re-converges. With the detector, adaptation pauses during double talk and the cancellation stays good. Real echo cancellers add a nonlinear processor that suppresses the small residual echo, and comfort noise to fill the resulting silence.

## 5. Game audio scene
A game positions each sound source around the listener and updates it as things move, block by block. For headphones, the main cues to direction and distance are:
- **Interaural time difference** (ITD): sound reaches the nearer ear first, by up to about 0.66 ms. For a spherical head of radius $a = 8.75$ cm, the Woodworth formula gives $\text{ITD} = \frac{a}{c}(\theta + \sin\theta)$ for a source at angle $\theta$ to the side.
- **Interaural level difference** (ILD): the head shadows the far ear, mostly at high frequencies. Modeled here as up to 6 dB less level plus a low-pass filter on the far ear.
- **Distance**: level falls by 6 dB per doubling of distance, the air absorbs high frequencies, and the ratio of direct sound to room reverb falls. The travel time to each ear changes as the source moves, which produces the **Doppler shift** automatically.

These cues do not tell front from back. That needs the direction-dependent filtering of the outer ear, captured in measured head-related transfer functions (HRTFs), which real game engines use.

The source below circles the listener once at 2 m, then flies away to 30 m. Parameters are computed once per 512-sample block, as in a game engine, and ramped linearly between blocks (`04_Real_time_Audio_Processing`). **Use headphones.**

In [ ]:
HEAD_RADIUS, SPEED_OF_SOUND, BLOCK = 0.0875, 343.0, 512

def scene_path(t):
    """Azimuth (radians, 0 = front, positive = right) and distance (m) over time."""
    circling = t < 6
    azimuth = np.where(circling, 2 * np.pi * t / 6, np.pi / 2)
    distance = np.where(circling, 2.0, 2.0 + 28 * np.clip((t - 6) / 4, 0, 1))
    return azimuth, distance

def render_scene(source, fs=FS):
    n = len(source)
    block_starts = np.arange(0, n + BLOCK, BLOCK)
    azimuth, distance = scene_path(block_starts / fs)               # parameters once per block...
    samples = np.arange(n)
    az = np.interp(samples, block_starts, np.unwrap(azimuth))       # ...ramped linearly in between
    dist = np.interp(samples, block_starts, distance)

    lateral = np.arcsin(np.sin(az))                                 # angle to the side, -90 to 90 degrees
    itd = HEAD_RADIUS / SPEED_OF_SOUND * (np.abs(lateral) + np.sin(np.abs(lateral)))
    travel = dist / SPEED_OF_SOUND                                  # propagation delay: gives Doppler
    shadow = 10 ** (-6 * np.abs(np.sin(lateral)) / 20)              # far ear: up to 6 dB quieter
    gain = 1 / np.maximum(dist, 0.5)
    ears = {}
    for ear, sign in (('left', -1), ('right', 1)):
        far_ear = sign * lateral < 0
        delay = travel + np.where(far_ear, itd, 0.0)
        y = np.interp(samples - delay * fs, samples, source, left=0.0, right=0.0)   # fractional delay
        y *= np.where(far_ear, shadow, 1.0) * gain
        # Head shadow on the far ear and air absorption, one-pole low-pass updated per block
        cutoff = np.where(far_ear, 2500 + 15000 * (1 - np.abs(np.sin(lateral))), 18000) / (1 + dist / 15)
        out, state = np.empty(n), 0.0
        for start in range(0, n, BLOCK):
            c = np.exp(-2 * np.pi * cutoff[start] / fs)
            segment, zf = signal.lfilter([1 - c], [1, -c], y[start:start + BLOCK], zi=[state])
            out[start:start + BLOCK], state = segment, zf[0]
        ears[ear] = out
    return ears['left'], ears['right'], itd, lateral

source = drum_loop(110, 6, FS, np.random.default_rng(3))[0][:10 * FS]
left, right, itd, lateral = render_scene(source)

# Room reverb at a fixed level: as the source moves away, the direct sound drops below it
t_rir = time_axis(1.2, FS)
reverb = [signal.fftconvolve(source, rng.normal(0, 1, len(t_rir)) * np.exp(-t_rir * 6.9 / 0.8))[:len(source)] for _ in range(2)]
reverb = [0.04 * r / np.sqrt(np.mean(r**2)) * np.sqrt(np.mean(source**2)) for r in reverb]
scene = np.vstack((left + reverb[0], right + reverb[1]))
scene *= 0.9 / np.max(np.abs(scene))

# Check the ITD: render white noise along the same path and cross-correlate the ears while the source
# is at the right (90 degrees, t = 1.5 s). Noise gives a single clear correlation peak; a short stretch
# of drums, with its repeating pattern, does not.
test_left, test_right, _, _ = render_scene(rng.normal(0, 0.3, len(source)))
at_right = slice(int(1.45 * FS), int(1.55 * FS))
lags = signal.correlation_lags(len(test_left[at_right]), len(test_right[at_right]))
measured = lags[np.argmax(signal.correlate(test_left[at_right], test_right[at_right]))]
print(f'Source at 90 degrees: model ITD {1000 * itd[int(1.5 * FS)]:.3f} ms, '
      f'measured left-ear lag {1000 * measured / FS:.3f} ms ({measured} samples)')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
t = np.arange(len(source)) / FS
axes[0].plot(t, np.degrees(lateral), label='angle to the side (degrees)')
axes[0].plot(t, 1000 * itd * np.sign(lateral) * 100, label='ITD (ms) x 100, positive = right ear first')
axes[0].set_xlabel('Time (s)')
axes[0].legend(fontsize=9)
for ch, name in zip(scene, ('left', 'right')):
    level = 10 * np.log10(np.convolve(ch**2, np.ones(4410) / 4410, mode='same') + 1e-9)
    axes[1].plot(t, level, label=name)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Level (dB)')
axes[1].set_title('Ear levels: circling, then flying away')
axes[1].legend()
plt.tight_layout()
plt.show()
play(scene, 'Game audio scene (headphones)')